In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Sat Apr 11 14:14:37 2026

@author: taylorroberson4
"""

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# =========================================================
# GLOBAL STYLE SETTINGS
# =========================================================
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 11,
    "axes.titlesize": 15,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.dpi": 120
})

# Color palette
TEAL = "#2A9D8F"
GREEN = "#4CAF50"
PURPLE = "#6A4C93"
BLUE = "#457B9D"
ORANGE = "#F4A261"
RED = "#E76F51"
LIGHT_GRAY = "#D9D9D9"

# =========================================================
# DATA
# =========================================================
reproduced_df = pd.DataFrame({
    "Task": [
        "Classification",
        "Clustering",
        "PairClassification",
        "Reranking",
        "Retrieval",
        "STS",
        "Summarization"
    ],
    "BAAI/bge-base-en-v1.5": [0.640596, 0.559759, 0.678872, 0.976689, 0.618503, 0.355123, 0.453604],
    "intfloat/e5-base-v2": [0.654155, 0.561392, 0.661632, 0.975959, 0.625210, 0.387627, 0.526124],
    "sentence-transformers/all-MiniLM-L6-v2": [0.579296, 0.537353, 0.582532, 0.973224, 0.568446, 0.308763, 0.363982]
})

paper_df = pd.DataFrame({
    "Model": [
        "BERT",
        "FinBERT",
        "instructor-base",
        "bge-large-en-v1.5",
        "AnglE-BERT",
        "gte-Qwen1.5-7B-instruct",
        "Echo",
        "bge-en-icl",
        "NV-Embed v2",
        "e5-mistral-7b-instruct",
        "text-embedding-3-small",
        "text-embedding-3-large",
        "voyage-3-large",
        "Fin-E5"
    ],
    "Avg": [0.3789, 0.4198, 0.3732, 0.3396, 0.3080, 0.3758, 0.4380, 0.3233, 0.3739, 0.3800, 0.3254, 0.3615, 0.4145, 0.4342]
})

retrieval_consistency_df = pd.DataFrame({
    "Model": [
        "sentence-transformers/all-MiniLM-L6-v2",
        "BAAI/bge-base-en-v1.5",
        "intfloat/e5-base-v2"
    ],
    "mean_retrieval": [0.568446, 0.618503, 0.625210],
    "std_retrieval": [0.304134, 0.310120, 0.313041],
    "min_retrieval": [0.10633, 0.13509, 0.15773],
    "max_retrieval": [0.97985, 0.98849, 0.98678]
})

repro_challenges_df = pd.DataFrame({
    "Challenge": [
        "Dependency Issues",
        "Inconsistent Metric Structures",
        "No Aggregation Pipeline",
        "Hardware Limits",
        "Limited Model Coverage",
        "Could Not Reproduce Fin-E5",
        "Missing Infrastructure Details",
        "No End-to-End Script"
    ],
    "Severity": [4, 4, 5, 5, 4, 5, 4, 5]
})

# =========================================================
# HELPER FUNCTION
# =========================================================
def finish_plot(filename=None):
    plt.grid(axis="y", linestyle="--", alpha=0.3)
    plt.tight_layout()
    if filename:
        plt.savefig(filename, bbox_inches="tight", dpi=300)
    plt.show()

# =========================================================
# VISUAL 1: GROUPED BAR CHART OF REPRODUCED RESULTS
# =========================================================
tasks = reproduced_df["Task"]
x = np.arange(len(tasks))
width = 0.25

plt.figure(figsize=(12, 6))
plt.bar(x - width, reproduced_df["BAAI/bge-base-en-v1.5"], width, label="BGE-base", color=TEAL)
plt.bar(x, reproduced_df["intfloat/e5-base-v2"], width, label="E5-base", color=GREEN)
plt.bar(x + width, reproduced_df["sentence-transformers/all-MiniLM-L6-v2"], width, label="MiniLM", color=PURPLE)

plt.xticks(x, tasks, rotation=25, ha="right")
plt.ylabel("Score")
plt.title("Reproduced FinMTEB Results by Task")
plt.legend(frameon=False)
finish_plot("reproduced_results_grouped_bar.png")

# =========================================================
# VISUAL 2: HEATMAP-STYLE PERFORMANCE MATRIX
# =========================================================
heatmap_df = reproduced_df.set_index("Task")
heatmap_values = heatmap_df.values

plt.figure(figsize=(8, 5))
im = plt.imshow(heatmap_values, aspect="auto", cmap="YlGnBu")

plt.xticks(np.arange(len(heatmap_df.columns)), ["BGE-base", "E5-base", "MiniLM"])
plt.yticks(np.arange(len(heatmap_df.index)), heatmap_df.index)

for i in range(len(heatmap_df.index)):
    for j in range(len(heatmap_df.columns)):
        plt.text(j, i, f"{heatmap_df.iloc[i, j]:.3f}", ha="center", va="center", color="black")

plt.title("Reproduced Results Heatmap")
plt.colorbar(im, label="Score")
plt.tight_layout()
plt.savefig("reproduced_results_heatmap.png", bbox_inches="tight", dpi=300)
plt.show()

# =========================================================
# VISUAL 3: AVERAGE SCORE ACROSS TASKS
# =========================================================
reproduced_avg_df = pd.DataFrame({
    "Model": ["BGE-base", "E5-base", "MiniLM"],
    "Average Score": [
        reproduced_df["BAAI/bge-base-en-v1.5"].mean(),
        reproduced_df["intfloat/e5-base-v2"].mean(),
        reproduced_df["sentence-transformers/all-MiniLM-L6-v2"].mean()
    ]
})

plt.figure(figsize=(8, 5))
bars = plt.bar(
    reproduced_avg_df["Model"],
    reproduced_avg_df["Average Score"],
    color=[TEAL, GREEN, PURPLE]
)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.005, f"{height:.3f}",
             ha="center", va="bottom")

plt.ylabel("Average Score")
plt.title("Average Performance Across Reproduced Tasks")
finish_plot("average_reproduced_scores.png")

# =========================================================
# VISUAL 4: RETRIEVAL CONSISTENCY (MEAN ± STD)
# =========================================================
plt.figure(figsize=(10, 6))
bars = plt.bar(
    ["MiniLM", "BGE-base", "E5-base"],
    retrieval_consistency_df["mean_retrieval"],
    yerr=retrieval_consistency_df["std_retrieval"],
    capsize=8,
    color=[PURPLE, TEAL, GREEN]
)

for bar, val in zip(bars, retrieval_consistency_df["mean_retrieval"]):
    plt.text(bar.get_x() + bar.get_width()/2, val + 0.02, f"{val:.3f}",
             ha="center", va="bottom")

plt.ylabel("Mean Retrieval Score")
plt.title("Retrieval Consistency Across Datasets (Mean ± Std. Dev.)")
finish_plot("retrieval_consistency_errorbar.png")

# =========================================================
# VISUAL 5: RETRIEVAL MIN-MAX RANGE PLOT
# =========================================================
plt.figure(figsize=(10, 6))

point_colors = [PURPLE, TEAL, GREEN]
labels = ["MiniLM", "BGE-base", "E5-base"]

for i in range(len(retrieval_consistency_df)):
    plt.plot(
        [i, i],
        [retrieval_consistency_df.loc[i, "min_retrieval"], retrieval_consistency_df.loc[i, "max_retrieval"]],
        color=LIGHT_GRAY,
        linewidth=6,
        solid_capstyle="round"
    )
    plt.scatter(i, retrieval_consistency_df.loc[i, "mean_retrieval"], color=point_colors[i], s=140, zorder=3)
    plt.scatter(i, retrieval_consistency_df.loc[i, "min_retrieval"], color=RED, s=60, zorder=3)
    plt.scatter(i, retrieval_consistency_df.loc[i, "max_retrieval"], color=BLUE, s=60, zorder=3)

plt.xticks(range(len(retrieval_consistency_df)), labels)
plt.ylabel("Retrieval Score")
plt.title("Retrieval Score Range Across Datasets")
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig("retrieval_min_max_range.png", bbox_inches="tight", dpi=300)
plt.show()

# =========================================================
# VISUAL 6: CONSISTENCY SCORE (MEAN / STD)
# =========================================================
retrieval_consistency_df["consistency_score"] = (
    retrieval_consistency_df["mean_retrieval"] / retrieval_consistency_df["std_retrieval"]
)

plt.figure(figsize=(8, 5))
bars = plt.bar(
    ["MiniLM", "BGE-base", "E5-base"],
    retrieval_consistency_df["consistency_score"],
    color=[PURPLE, TEAL, GREEN]
)

for bar, val in zip(bars, retrieval_consistency_df["consistency_score"]):
    plt.text(bar.get_x() + bar.get_width()/2, val + 0.02, f"{val:.2f}",
             ha="center", va="bottom")

plt.ylabel("Consistency Score")
plt.title("Retrieval Stability Score (Mean / Std. Dev.)")
finish_plot("retrieval_stability_score.png")

# =========================================================
# VISUAL 7: REPRODUCIBILITY CHALLENGES
# =========================================================
plt.figure(figsize=(11, 7))
bars = plt.barh(
    repro_challenges_df["Challenge"],
    repro_challenges_df["Severity"],
    color=[ORANGE, ORANGE, RED, RED, ORANGE, RED, ORANGE, RED]
)

for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.05, bar.get_y() + bar.get_height()/2, f"{width}",
             va="center")

plt.xlabel("Severity")
plt.title("Practical Reproducibility Challenges in FinMTEB")
plt.grid(axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig("reproducibility_challenges.png", bbox_inches="tight", dpi=300)
plt.show()

# =========================================================
# VISUAL 8: CONCEPTUAL VS PRACTICAL REPRODUCIBILITY
# =========================================================
conceptual_vs_practical_df = pd.DataFrame({
    "Type": ["Conceptual Reproducibility", "Practical Reproducibility"],
    "Score": [9, 5]
})

plt.figure(figsize=(8, 5))
bars = plt.bar(
    conceptual_vs_practical_df["Type"],
    conceptual_vs_practical_df["Score"],
    color=[BLUE, RED]
)

for bar, val in zip(bars, conceptual_vs_practical_df["Score"]):
    plt.text(bar.get_x() + bar.get_width()/2, val + 0.15, f"{val}/10",
             ha="center", va="bottom")

plt.ylabel("Score")
plt.title("FinMTEB Reproducibility Assessment")
finish_plot("conceptual_vs_practical_reproducibility.png")

# =========================================================
# VISUAL 9: PAPER BENCHMARK AVERAGES
# =========================================================
paper_sorted = paper_df.sort_values("Avg", ascending=False)

highlight_colors = []
for model in paper_sorted["Model"]:
    if model == "Fin-E5":
        highlight_colors.append(RED)
    elif model == "voyage-3-large":
        highlight_colors.append(PURPLE)
    elif model == "bge-large-en-v1.5":
        highlight_colors.append(TEAL)
    else:
        highlight_colors.append(BLUE)

plt.figure(figsize=(12, 6))
plt.bar(paper_sorted["Model"], paper_sorted["Avg"], color=highlight_colors)
plt.xticks(rotation=55, ha="right")
plt.ylabel("Average Score")
plt.title("Paper Benchmark Average Scores")
finish_plot("paper_benchmark_averages.png")

# =========================================================
# VISUAL 10: CONTEXTUAL COMPARISON
# =========================================================
context_df = pd.DataFrame({
    "Model": [
        "BGE-base\n(reproduced)",
        "E5-base\n(reproduced)",
        "MiniLM\n(reproduced)",
        "bge-large\n(paper)",
        "Fin-E5\n(paper)",
        "voyage-3-large\n(paper)"
    ],
    "Average": [
        reproduced_df["BAAI/bge-base-en-v1.5"].mean(),
        reproduced_df["intfloat/e5-base-v2"].mean(),
        reproduced_df["sentence-transformers/all-MiniLM-L6-v2"].mean(),
        paper_df.loc[paper_df["Model"] == "bge-large-en-v1.5", "Avg"].values[0],
        paper_df.loc[paper_df["Model"] == "Fin-E5", "Avg"].values[0],
        paper_df.loc[paper_df["Model"] == "voyage-3-large", "Avg"].values[0]
    ]
})

plt.figure(figsize=(11, 6))
bars = plt.bar(
    context_df["Model"],
    context_df["Average"],
    color=[TEAL, GREEN, PURPLE, BLUE, RED, ORANGE]
)

for bar, val in zip(bars, context_df["Average"]):
    plt.text(bar.get_x() + bar.get_width()/2, val + 0.005, f"{val:.3f}",
             ha="center", va="bottom", fontsize=9)

plt.ylabel("Average Score")
plt.title("Contextual Comparison: Reproduced Models vs Paper Models")
finish_plot("contextual_comparison.png")

# =========================================================
# OPTIONAL: PRINT TABLES
# =========================================================
print("\nReproduced Results:")
print(reproduced_df.round(4))

print("\nAverage Reproduced Scores:")
print(reproduced_avg_df.round(4))

print("\nRetrieval Consistency:")
print(retrieval_consistency_df.round(4))

print("\nReproducibility Challenges:")
print(repro_challenges_df)